# Text Denoising and Correction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/text-denoising.ipynb)

This notebook explores techniques for automatically correcting noisy or corrupted text. We'll implement multiple approaches:

1. **Understanding Text Noise** - Types of corruption and their real-world sources
2. **Synthetic Data Generation** - Creating training data with controlled noise
3. **Baseline Methods** - Edit distance and dictionary-based correction
4. **Sequence-to-Sequence Models** - RNN with attention for denoising
5. **Transformer Architecture** - Modern approach with self-attention
6. **Pre-trained Models** - Fine-tuning T5 for correction tasks
7. **Evaluation** - Character/Word Error Rate metrics

**Learning Objectives:**
- Understand different types of text corruption
- Implement noise injection for data augmentation
- Build seq2seq models for text correction
- Compare baseline vs. learned approaches
- Evaluate correction quality with standard metrics

In [ ]:
# Configuration dictionary# All hyperparameters and settings for the notebook are defined hereCONFIG = {    # General settings    'seed': 42,        # Dataset settings    'dataset_id': 'names',    'train_split': 0.9,    'val_split': 0.1,    'noise_prob': 0.2,  # Probability of corrupting a character        # Model architecture    'embedding_dim': 64,    'hidden_size': 256,    'num_layers': 2,    'dropout': 0.2,        # Training settings    'batch_size': 128,    'learning_rate': 0.001,    'num_epochs': 1,    'gradient_clip': 1.0,        # Sampling settings    'temperature': 1.0,    'max_length': 20,}print("Configuration loaded:")for key, value in CONFIG.items():    print(f"  {key}: {value}")

## Part 1: Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
import string
from collections import Counter
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
from tqdm.auto import tqdm
import math
import os
from aiml_notebooks import get_device, set_seed

# Set random seeds for reproducibility
set_seed(42)

# Device configuration (safe mode for Transformer compatibility)
device = get_device(prefer_cpu=True)

print(f"PyTorch version: {torch.__version__}")

## Part 2: Understanding Text Noise

Text corruption can occur in many ways:

1. **Character Substitution**: `hello` → `hEllo` (wrong character)
2. **Character Insertion**: `hello` → `hellow` (extra character)
3. **Character Deletion**: `hello` → `helo` (missing character)
4. **Transposition**: `hello` → `hlelo` (swapped adjacent characters)

These errors commonly arise from:
- OCR (Optical Character Recognition) errors
- Keyboard typos
- Speech-to-text mistakes
- Data transmission errors
- Historical document degradation

Our goal is to learn a model that can reverse these corruptions.

## Part 3: Noise Injection Functions

In [ ]:
class TextNoiser:
    """Applies various types of noise to clean text."""
    
    def __init__(self, 
                 sub_prob: float = 0.1,  # Substitution probability
                 ins_prob: float = 0.1,  # Insertion probability
                 del_prob: float = 0.1,  # Deletion probability
                 swap_prob: float = 0.05):  # Transposition probability
        self.sub_prob = sub_prob
        self.ins_prob = ins_prob
        self.del_prob = del_prob
        self.swap_prob = swap_prob
        
        # Characters to use for substitution/insertion
        self.chars = string.ascii_lowercase + string.ascii_uppercase + ' '
    
    def add_noise(self, text: str) -> str:
        """Apply random noise to text."""
        chars = list(text)
        i = 0
        
        while i < len(chars):
            # Character substitution
            if random.random() < self.sub_prob:
                chars[i] = random.choice(self.chars)
            
            # Character deletion
            if random.random() < self.del_prob:
                chars.pop(i)
                continue
            
            # Character insertion
            if random.random() < self.ins_prob:
                chars.insert(i, random.choice(self.chars))
                i += 1
            
            # Character swap (transposition)
            if i < len(chars) - 1 and random.random() < self.swap_prob:
                chars[i], chars[i + 1] = chars[i + 1], chars[i]
                i += 1
            
            i += 1
        
        return ''.join(chars)

# Test the noiser
noiser = TextNoiser(sub_prob=0.15, ins_prob=0.1, del_prob=0.1, swap_prob=0.05)

test_sentences = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning is fascinating",
    "Neural networks can denoise text"
]

print("Examples of noise injection:\n")
for sent in test_sentences:
    noisy = noiser.add_noise(sent)
    print(f"Clean:  {sent}")
    print(f"Noisy:  {noisy}")
    print()

## Part 4: Sample Dataset

For this educational notebook, we'll use a small corpus of common English sentences. In practice, you'd use:
- Wikipedia articles
- Book corpora (e.g., BookCorpus)
- News articles
- Domain-specific text

In [ ]:
# Small sample corpus for demonstration
SAMPLE_CORPUS = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning is a subset of artificial intelligence",
    "Neural networks learn patterns from data",
    "Text denoising removes corruption from sentences",
    "Deep learning models can correct spelling errors",
    "Natural language processing enables computers to understand text",
    "Training data should be diverse and representative",
    "Attention mechanisms help models focus on relevant information",
    "Transformers use self-attention to process sequences",
    "The cat sat on the mat and watched the birds",
    "Python is a popular programming language for data science",
    "Gradient descent optimizes neural network parameters",
    "Backpropagation computes gradients for learning",
    "Overfitting occurs when models memorize training data",
    "Regularization techniques prevent overfitting",
    "Cross-validation helps evaluate model performance",
    "The weather is beautiful today with clear blue skies",
    "Books provide knowledge and entertainment to readers",
    "Music brings joy and inspiration to many people",
    "Exercise and healthy eating contribute to wellbeing",
]

# Expand corpus by generating variations
expanded_corpus = SAMPLE_CORPUS * 50  # 1000 samples for training

print(f"Corpus size: {len(expanded_corpus)} sentences")
print(f"Sample: {expanded_corpus[0]}")

## Part 5: Character-Level Tokenization

For text denoising, we work at the character level because:
- Errors occur at character granularity
- We need to handle misspelled words
- Character-level models can generalize to unseen words

In [ ]:
class CharTokenizer:
    """Simple character-level tokenizer."""
    
    def __init__(self, texts: List[str]):
        # Build vocabulary from texts
        chars = set(''.join(texts))
        
        # Special tokens
        self.pad_token = '<PAD>'
        self.sos_token = '<SOS>'  # Start of sequence
        self.eos_token = '<EOS>'  # End of sequence
        self.unk_token = '<UNK>'  # Unknown character
        
        # Create vocabulary
        self.vocab = [self.pad_token, self.sos_token, self.eos_token, self.unk_token] + sorted(list(chars))
        self.char2idx = {c: i for i, c in enumerate(self.vocab)}
        self.idx2char = {i: c for i, c in enumerate(self.vocab)}
        
        self.vocab_size = len(self.vocab)
        self.pad_idx = self.char2idx[self.pad_token]
        self.sos_idx = self.char2idx[self.sos_token]
        self.eos_idx = self.char2idx[self.eos_token]
        self.unk_idx = self.char2idx[self.unk_token]
    
    def encode(self, text: str) -> List[int]:
        """Convert text to token IDs."""
        return [self.char2idx.get(c, self.unk_idx) for c in text]
    
    def decode(self, ids: List[int]) -> str:
        """Convert token IDs to text."""
        return ''.join([self.idx2char[i] for i in ids if i not in [self.pad_idx, self.sos_idx, self.eos_idx]])

# Create tokenizer
tokenizer = CharTokenizer(expanded_corpus)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Vocabulary: {tokenizer.vocab[:20]}...")  # Show first 20 tokens

# Test encoding/decoding
test_text = "Hello World"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"\nOriginal: {test_text}")
print(f"Encoded:  {encoded}")
print(f"Decoded:  {decoded}")

## Part 6: Dataset and DataLoader

In [ ]:
class DenoisingDataset(Dataset):
    """Dataset for text denoising."""
    
    def __init__(self, texts: List[str], tokenizer: CharTokenizer, noiser: TextNoiser):
        self.texts = texts
        self.tokenizer = tokenizer
        self.noiser = noiser
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        clean_text = self.texts[idx]
        noisy_text = self.noiser.add_noise(clean_text)
        
        # Encode
        src = self.tokenizer.encode(noisy_text)
        tgt = self.tokenizer.encode(clean_text)
        
        return torch.tensor(src), torch.tensor(tgt)

def collate_fn(batch):
    """Collate function to pad sequences in batch."""
    src_batch, tgt_batch = zip(*batch)
    
    # Pad sequences
    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=tokenizer.pad_idx)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=tokenizer.pad_idx)
    
    return src_padded, tgt_padded

# Create datasets
train_size = int(0.8 * len(expanded_corpus))
val_size = len(expanded_corpus) - train_size

train_texts = expanded_corpus[:train_size]
val_texts = expanded_corpus[train_size:]

train_dataset = DenoisingDataset(train_texts, tokenizer, noiser)
val_dataset = DenoisingDataset(val_texts, tokenizer, noiser)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Batch size: {batch_size}")

# Show a sample batch
src, tgt = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"Source (noisy): {src.shape}")
print(f"Target (clean): {tgt.shape}")

# Decode first sample
print(f"\nFirst sample:")
print(f"Noisy:  {tokenizer.decode(src[0].tolist())}")
print(f"Clean:  {tokenizer.decode(tgt[0].tolist())}")

## Part 7: Baseline - Edit Distance

Before building neural models, let's implement a simple baseline using edit distance (Levenshtein distance).

**Idea**: For each noisy word, find the closest word in a dictionary using edit distance.

In [ ]:
def edit_distance(s1: str, s2: str) -> int:
    """Compute Levenshtein edit distance between two strings."""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    # Initialize base cases
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    
    # Fill DP table
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],      # deletion
                    dp[i][j - 1],      # insertion
                    dp[i - 1][j - 1]   # substitution
                )
    
    return dp[m][n]

class EditDistanceCorrector:
    """Spell checker using edit distance and a dictionary."""
    
    def __init__(self, dictionary: List[str]):
        # Build word dictionary (lowercase)
        self.dictionary = set()
        for text in dictionary:
            self.dictionary.update(text.lower().split())
        print(f"Dictionary size: {len(self.dictionary)} unique words")
    
    def correct_word(self, word: str, max_distance: int = 2) -> str:
        """Find closest word in dictionary."""
        if word.lower() in self.dictionary:
            return word
        
        # Find closest match
        candidates = []
        for dict_word in self.dictionary:
            dist = edit_distance(word.lower(), dict_word)
            if dist <= max_distance:
                candidates.append((dist, dict_word))
        
        if candidates:
            candidates.sort()
            return candidates[0][1]
        return word
    
    def correct_text(self, text: str) -> str:
        """Correct all words in text."""
        words = text.split()
        corrected = [self.correct_word(w) for w in words]
        return ' '.join(corrected)

# Create corrector
corrector = EditDistanceCorrector(SAMPLE_CORPUS)

# Test on noisy examples
test_cases = [
    "Teh qick brown fox",
    "Machne lerning is fasinating",
    "Nerual netwrks can denose text"
]

print("\nEdit distance baseline results:\n")
for noisy in test_cases:
    corrected = corrector.correct_text(noisy)
    print(f"Noisy:     {noisy}")
    print(f"Corrected: {corrected}")
    print()

## Part 8: Evaluation Metrics

We'll use two standard metrics:

1. **Character Error Rate (CER)**: Edit distance at character level
   - CER = (substitutions + insertions + deletions) / total characters
   - Lower is better (0 = perfect)

2. **Word Error Rate (WER)**: Edit distance at word level
   - WER = (substitutions + insertions + deletions) / total words
   - Lower is better (0 = perfect)

In [ ]:
def character_error_rate(predicted: str, target: str) -> float:
    """Compute Character Error Rate (CER)."""
    if len(target) == 0:
        return 0.0 if len(predicted) == 0 else 1.0
    return edit_distance(predicted, target) / len(target)

def word_error_rate(predicted: str, target: str) -> float:
    """Compute Word Error Rate (WER)."""
    pred_words = predicted.split()
    tgt_words = target.split()
    
    if len(tgt_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    
    return edit_distance(' '.join(pred_words), ' '.join(tgt_words)) / len(tgt_words)

# Test metrics
examples = [
    ("hello world", "hello world"),  # Perfect match
    ("helo world", "hello world"),   # 1 char error
    ("helo wrld", "hello world"),    # 2 char errors
]

print("Metric examples:\n")
for pred, tgt in examples:
    cer = character_error_rate(pred, tgt)
    wer = word_error_rate(pred, tgt)
    print(f"Predicted: '{pred}'")
    print(f"Target:    '{tgt}'")
    print(f"CER: {cer:.3f}, WER: {wer:.3f}")
    print()

## Part 9: Seq2Seq with Attention - Theory

**Sequence-to-Sequence (Seq2Seq)** models are designed for transforming one sequence into another.

**Architecture:**
1. **Encoder**: Processes noisy input and creates context representation
2. **Attention**: Allows decoder to focus on relevant parts of input
3. **Decoder**: Generates clean output character by character

**Why Attention?**
- Helps model align input/output characters
- Allows focusing on local context during correction
- Improves long-sequence performance

**Training:**
- Teacher forcing: Feed ground truth during training
- Cross-entropy loss at each output position

## Part 10: Seq2Seq Implementation - Encoder

In [ ]:
class Encoder(nn.Module):
    """Bidirectional LSTM encoder."""
    
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_layers: int = 1, dropout: float = 0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=tokenizer.pad_idx)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, embed_dim)
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: (batch, seq_len, 2 * hidden_dim)
        # hidden: (2 * num_layers, batch, hidden_dim)
        return outputs, hidden, cell

# Test encoder
encoder = Encoder(
    vocab_size=tokenizer.vocab_size,
    embed_dim=64,
    hidden_dim=128,
    num_layers=CONFIG['num_layers'],
    dropout=0.1
).to(device)

test_input = torch.randint(0, tokenizer.vocab_size, (2, 10)).to(device)  # (batch=2, seq_len=10)
outputs, hidden, cell = encoder(test_input)
print(f"Encoder output shape: {outputs.shape}")  # (2, 10, 256)
print(f"Hidden state shape: {hidden.shape}")     # (4, 2, 128)

## Part 11: Attention Mechanism

In [ ]:
class Attention(nn.Module):
    """Bahdanau attention mechanism."""
    
    def __init__(self, hidden_dim: int, encoder_hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim + encoder_hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, hidden, encoder_outputs):
        # hidden: (batch, hidden_dim)
        # encoder_outputs: (batch, seq_len, encoder_hidden_dim)
        
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        
        # Repeat hidden state for each source position
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch, src_len, hidden_dim)
        
        # Concatenate and compute attention scores
        energy = torch.tanh(self.attn(torch.cat([hidden, encoder_outputs], dim=2)))  # (batch, src_len, hidden_dim)
        attention = self.v(energy).squeeze(2)  # (batch, src_len)
        
        # Softmax to get attention weights
        attn_weights = F.softmax(attention, dim=1)  # (batch, src_len)
        
        return attn_weights

## Part 12: Seq2Seq Decoder

In [ ]:
class Decoder(nn.Module):
    """LSTM decoder with attention."""
    
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, encoder_hidden_dim: int, num_layers: int = 1, dropout: float = 0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.attention = Attention(hidden_dim, encoder_hidden_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=tokenizer.pad_idx)
        self.lstm = nn.LSTM(
            embed_dim + encoder_hidden_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, input, hidden, cell, encoder_outputs):
        # input: (batch)
        # hidden: (num_layers, batch, hidden_dim)
        # encoder_outputs: (batch, src_len, encoder_hidden_dim)
        
        input = input.unsqueeze(1)  # (batch, 1)
        embedded = self.dropout(self.embedding(input))  # (batch, 1, embed_dim)
        
        # Compute attention
        attn_weights = self.attention(hidden[-1], encoder_outputs)  # (batch, src_len)
        attn_weights = attn_weights.unsqueeze(1)  # (batch, 1, src_len)
        
        # Apply attention to encoder outputs
        context = torch.bmm(attn_weights, encoder_outputs)  # (batch, 1, encoder_hidden_dim)
        
        # Concatenate embedding and context
        lstm_input = torch.cat([embedded, context], dim=2)  # (batch, 1, embed_dim + encoder_hidden_dim)
        
        # LSTM step
        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        
        # Prediction
        prediction = self.fc_out(output.squeeze(1))  # (batch, vocab_size)
        
        return prediction, hidden, cell, attn_weights.squeeze(1)

## Part 13: Complete Seq2Seq Model

In [ ]:
class Seq2Seq(nn.Module):
    """Complete sequence-to-sequence model."""
    
    def __init__(self, encoder: Encoder, decoder: Decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, tgt, teacher_forcing_ratio: float = 0.5):
        # src: (batch, src_len)
        # tgt: (batch, tgt_len)
        
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        vocab_size = self.decoder.vocab_size
        
        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(self.device)
        
        # Encode
        encoder_outputs, hidden, cell = self.encoder(src)
        
        # Combine bidirectional hidden states
        # hidden: (2 * num_layers, batch, hidden_dim) -> (num_layers, batch, hidden_dim)
        hidden = hidden.view(self.encoder.lstm.num_layers, 2, batch_size, -1)
        hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)  # Concatenate forward/backward
        cell = cell.view(self.encoder.lstm.num_layers, 2, batch_size, -1)
        cell = torch.cat([cell[:, 0, :, :], cell[:, 1, :, :]], dim=2)
        
        # First input to decoder is SOS token
        input = torch.full((batch_size,), tokenizer.sos_idx, dtype=torch.long).to(self.device)
        
        # Decode
        for t in range(tgt_len):
            output, hidden, cell, _ = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t] = output
            
            # Teacher forcing: use ground truth as next input
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = tgt[:, t] if teacher_force else top1
        
        return outputs
    
    def generate(self, src, max_len: int = 100):
        """Generate output without teacher forcing."""
        self.eval()
        with torch.no_grad():
            batch_size = src.shape[0]
            
            # Encode
            encoder_outputs, hidden, cell = self.encoder(src)
            
            # Combine bidirectional states
            hidden = hidden.view(self.encoder.lstm.num_layers, 2, batch_size, -1)
            hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)
            cell = cell.view(self.encoder.lstm.num_layers, 2, batch_size, -1)
            cell = torch.cat([cell[:, 0, :, :], cell[:, 1, :, :]], dim=2)
            
            # Start with SOS
            input = torch.full((batch_size,), tokenizer.sos_idx, dtype=torch.long).to(self.device)
            
            outputs = []
            for _ in range(max_len):
                output, hidden, cell, _ = self.decoder(input, hidden, cell, encoder_outputs)
                top1 = output.argmax(1)
                outputs.append(top1)
                
                # Stop if all sequences generated EOS
                if (top1 == tokenizer.eos_idx).all():
                    break
                
                input = top1
            
            return torch.stack(outputs, dim=1)  # (batch, seq_len)

# Create model
embed_dim = 128
hidden_dim = 256
encoder_hidden_dim = 256 * 2  # Bidirectional

encoder = Encoder(
    vocab_size=tokenizer.vocab_size,
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
    num_layers=CONFIG['num_layers'],
    dropout=0.3
).to(device)

decoder = Decoder(
    vocab_size=tokenizer.vocab_size,
    embed_dim=embed_dim,
    hidden_dim=hidden_dim * 2,  # Match encoder's output dim
    encoder_hidden_dim=encoder_hidden_dim,
    num_layers=CONFIG['num_layers'],
    dropout=0.3
).to(device)

seq2seq_model = Seq2Seq(encoder, decoder, device).to(device)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Seq2Seq model has {count_parameters(seq2seq_model):,} trainable parameters")

## Part 14: Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, clip: float = 1.0):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for src, tgt in tqdm(loader, desc="Training"):
        src, tgt = src.to(device), tgt.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        output = model(src, tgt)
        
        # Reshape for loss computation
        output_dim = output.shape[-1]
        output = output.view(-1, output_dim)
        tgt = tgt.view(-1)
        
        # Compute loss
        loss = criterion(output, tgt)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    """Evaluate model."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            
            output = model(src, tgt, teacher_forcing_ratio=0)  # No teacher forcing during eval
            
            output_dim = output.shape[-1]
            output = output.view(-1, output_dim)
            tgt = tgt.view(-1)
            
            loss = criterion(output, tgt)
            total_loss += loss.item()
    
    return total_loss / len(loader)

# Training setup
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)
optimizer = torch.optim.Adam(seq2seq_model.parameters(), lr=CONFIG['learning_rate'])

num_epochs=CONFIG['num_epochs']
train_losses = []
val_losses = []

print("Training Seq2Seq model...\n")
for epoch in range(num_epochs):
    train_loss = train_epoch(seq2seq_model, train_loader, optimizer, criterion)
    val_loss = evaluate(seq2seq_model, val_loader, criterion)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")

print("\nTraining complete!")

## Part 15: Visualize Training

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Seq2Seq Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Part 16: Test Seq2Seq Model

In [ ]:
def denoise_text(model, text: str, tokenizer: CharTokenizer) -> str:
    """Denoise a single text string."""
    model.eval()
    
    # Encode input
    src = torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(device)  # (1, seq_len)
    
    # Generate output
    output = model.generate(src, max_len=len(text) * 2)
    
    # Decode
    denoised = tokenizer.decode(output[0].tolist())
    return denoised

# Test on examples
test_sentences = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning is fascinating",
    "Neural networks can denoise text",
    "Deep learning requires large datasets",
    "Attention mechanisms improve performance"
]

print("Seq2Seq Denoising Results:\n")
for clean_text in test_sentences:
    # Add noise
    noisy_text = noiser.add_noise(clean_text)
    
    # Denoise
    denoised = denoise_text(seq2seq_model, noisy_text, tokenizer)
    
    # Compute metrics
    cer = character_error_rate(denoised, clean_text)
    wer = word_error_rate(denoised, clean_text)
    
    print(f"Clean:    {clean_text}")
    print(f"Noisy:    {noisy_text}")
    print(f"Denoised: {denoised}")
    print(f"CER: {cer:.3f}, WER: {wer:.3f}")
    print()

## Part 17: Transformer Architecture - Theory

**Transformers** have become the dominant architecture for sequence tasks.

**Key Differences from RNNs:**
- **Parallelization**: Process entire sequence at once (vs. sequential)
- **Self-attention**: Every position attends to all others
- **Positional encoding**: Inject position information explicitly
- **Multi-head attention**: Multiple attention patterns simultaneously

**Architecture Components:**
1. **Input Embedding** + Positional Encoding
2. **Multi-Head Self-Attention**: Learn relationships between all positions
3. **Feed-Forward Network**: Process each position independently
4. **Layer Normalization**: Stabilize training
5. **Residual Connections**: Enable deep networks

**For Denoising:**
- Encoder-decoder structure (like seq2seq)
- Encoder processes noisy text
- Decoder generates clean text with cross-attention to encoder

## Part 18: Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Register as buffer (not a parameter)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:x.size(1), :]
        return self.dropout(x)

# Visualize positional encodings
pe = PositionalEncoding(d_model=128, max_len=100)
encodings = pe.pe.numpy()

plt.figure(figsize=(12, 4))
plt.imshow(encodings.T, aspect='auto', cmap='RdBu')
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Positional Encoding Visualization')
plt.colorbar()
plt.show()

## Part 19: Transformer Model

In [ ]:
class TransformerDenoiser(nn.Module):
    """Transformer model for text denoising."""
    
    def __init__(self, 
                 vocab_size: int,
                 d_model: int = 256,
                 nhead: int = 8,
                 num_encoder_layers: int = 3,
                 num_decoder_layers: int = 3,
                 dim_feedforward: int = 1024,
                 dropout: float = 0.1,
                 max_len: int = 200):
        super().__init__()
        
        self.d_model = d_model
        self.vocab_size = vocab_size
        
        # Embeddings
        self.encoder_embedding = nn.Embedding(vocab_size, d_model, padding_idx=tokenizer.pad_idx)
        self.decoder_embedding = nn.Embedding(vocab_size, d_model, padding_idx=tokenizer.pad_idx)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)
        
        # Transformer
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        # Output projection
        self.fc_out = nn.Linear(d_model, vocab_size)
        
        self.init_weights()
    
    def init_weights(self):
        """Initialize parameters."""
        initrange = 0.1
        self.encoder_embedding.weight.data.uniform_(-initrange, initrange)
        self.decoder_embedding.weight.data.uniform_(-initrange, initrange)
        self.fc_out.bias.data.zero_()
        self.fc_out.weight.data.uniform_(-initrange, initrange)
    
    def generate_square_subsequent_mask(self, sz: int):
        """Generate causal mask for decoder."""
        mask = torch.triu(torch.ones(sz, sz), diagonal=1).bool()
        return mask
    
    def forward(self, src, tgt):
        # src: (batch, src_len)
        # tgt: (batch, tgt_len)
        
        # Create masks
        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
        src_padding_mask = (src == tokenizer.pad_idx)
        tgt_padding_mask = (tgt == tokenizer.pad_idx)
        
        # Embed and add positional encoding
        src_emb = self.pos_encoder(self.encoder_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.decoder_embedding(tgt) * math.sqrt(self.d_model))
        
        # Transformer forward pass
        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )
        
        # Project to vocabulary
        return self.fc_out(output)
    
    def generate(self, src, max_len: int = 100):
        """Greedy decoding."""
        self.eval()
        with torch.no_grad():
            batch_size = src.size(0)
            
            # Start with SOS token
            tgt = torch.full((batch_size, 1), tokenizer.sos_idx, dtype=torch.long).to(src.device)
            
            for _ in range(max_len):
                output = self.forward(src, tgt)
                next_token = output[:, -1, :].argmax(dim=-1).unsqueeze(1)
                tgt = torch.cat([tgt, next_token], dim=1)
                
                # Stop if all sequences generated EOS
                if (next_token == tokenizer.eos_idx).all():
                    break
            
            return tgt[:, 1:]  # Remove SOS token

# Create transformer model
transformer_model = TransformerDenoiser(
    vocab_size=tokenizer.vocab_size,
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=1024,
    dropout=0.1
).to(device)

print(f"Transformer model has {count_parameters(transformer_model):,} trainable parameters")

## Part 20: Train Transformer Model

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=0.0001)

num_epochs=CONFIG['num_epochs']
transformer_train_losses = []
transformer_val_losses = []

print("Training Transformer model...\n")
for epoch in range(num_epochs):
    # Train
    transformer_model.train()
    total_loss = 0
    
    for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
        src, tgt = src.to(device), tgt.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        output = transformer_model(src, tgt)
        
        # Reshape and compute loss
        output = output.reshape(-1, tokenizer.vocab_size)
        tgt = tgt.reshape(-1)
        
        loss = criterion(output, tgt)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])
        optimizer.step()
        
        total_loss += loss.item()
    
    train_loss = total_loss / len(train_loader)
    
    # Validate
    transformer_model.eval()
    total_val_loss = 0
    
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            output = transformer_model(src, tgt)
            output = output.reshape(-1, tokenizer.vocab_size)
            tgt = tgt.reshape(-1)
            loss = criterion(output, tgt)
            total_val_loss += loss.item()
    
    val_loss = total_val_loss / len(val_loader)
    
    transformer_train_losses.append(train_loss)
    transformer_val_losses.append(val_loss)
    
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")

print("\nTraining complete!")

## Part 21: Compare Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Seq2Seq
axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Seq2Seq Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Transformer
axes[1].plot(transformer_train_losses, label='Train Loss')
axes[1].plot(transformer_val_losses, label='Val Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Transformer Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 22: Test Transformer Model

In [ ]:
def denoise_text_transformer(model, text: str, tokenizer: CharTokenizer) -> str:
    """Denoise text using transformer model."""
    model.eval()
    
    # Encode
    src = torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(device)
    
    # Generate
    output = model.generate(src, max_len=len(text) * 2)
    
    # Decode
    return tokenizer.decode(output[0].tolist())

# Test transformer
print("Transformer Denoising Results:\n")
for clean_text in test_sentences:
    noisy_text = noiser.add_noise(clean_text)
    denoised = denoise_text_transformer(transformer_model, noisy_text, tokenizer)
    
    cer = character_error_rate(denoised, clean_text)
    wer = word_error_rate(denoised, clean_text)
    
    print(f"Clean:    {clean_text}")
    print(f"Noisy:    {noisy_text}")
    print(f"Denoised: {denoised}")
    print(f"CER: {cer:.3f}, WER: {wer:.3f}")
    print()

## Part 23: Model Comparison

In [ ]:
def evaluate_model(model, test_texts: List[str], noiser: TextNoiser, is_transformer: bool = False):
    """Evaluate model on test set."""
    cers = []
    wers = []
    
    for clean_text in tqdm(test_texts, desc="Evaluating"):
        noisy_text = noiser.add_noise(clean_text)
        
        if is_transformer:
            denoised = denoise_text_transformer(model, noisy_text, tokenizer)
        else:
            denoised = denoise_text(model, noisy_text, tokenizer)
        
        cers.append(character_error_rate(denoised, clean_text))
        wers.append(word_error_rate(denoised, clean_text))
    
    return np.mean(cers), np.mean(wers)

# Evaluate both models
test_texts = val_texts[:50]  # Sample of validation set

seq2seq_cer, seq2seq_wer = evaluate_model(seq2seq_model, test_texts, noiser, is_transformer=False)
transformer_cer, transformer_wer = evaluate_model(transformer_model, test_texts, noiser, is_transformer=True)

# Display results
print("\nModel Comparison:\n")
print(f"{'Model':<15} {'CER':<10} {'WER':<10}")
print("-" * 35)
print(f"{'Seq2Seq':<15} {seq2seq_cer:<10.4f} {seq2seq_wer:<10.4f}")
print(f"{'Transformer':<15} {transformer_cer:<10.4f} {transformer_wer:<10.4f}")

# Visualize comparison
models = ['Seq2Seq', 'Transformer']
cers = [seq2seq_cer, transformer_cer]
wers = [seq2seq_wer, transformer_wer]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, cers, width, label='CER')
ax.bar(x + width/2, wers, width, label='WER')

ax.set_ylabel('Error Rate')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()

## Part 24: Interactive Demo

In [ ]:
def interactive_demo():
    """Interactive text denoising demo."""
    print("=" * 60)
    print("Text Denoising Demo")
    print("=" * 60)
    print("Enter text to denoise (or 'quit' to exit)\n")
    
    while True:
        text = input("Input: ").strip()
        
        if text.lower() == 'quit':
            break
        
        if not text:
            continue
        
        # Add noise
        noisy = noiser.add_noise(text)
        
        # Denoise with both models
        seq2seq_output = denoise_text(seq2seq_model, noisy, tokenizer)
        transformer_output = denoise_text_transformer(transformer_model, noisy, tokenizer)
        
        print(f"\nOriginal:    {text}")
        print(f"Noisy:       {noisy}")
        print(f"Seq2Seq:     {seq2seq_output}")
        print(f"Transformer: {transformer_output}")
        print()

# Uncomment to run interactive demo
# interactive_demo()

## Part 25: Attention Visualization

Let's visualize what the attention mechanism learns.

In [ ]:
def get_attention_weights(model, src, tgt):
    """Extract attention weights from seq2seq model."""
    model.eval()
    
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src)
        
        # Combine bidirectional states
        batch_size = src.shape[0]
        hidden = hidden.view(model.encoder.lstm.num_layers, 2, batch_size, -1)
        hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)
        cell = cell.view(model.encoder.lstm.num_layers, 2, batch_size, -1)
        cell = torch.cat([cell[:, 0, :, :], cell[:, 1, :, :]], dim=2)
        
        attention_weights = []
        input_token = torch.tensor([tokenizer.sos_idx]).to(device)
        
        for t in range(tgt.size(1)):
            _, hidden, cell, attn = model.decoder(input_token, hidden, cell, encoder_outputs)
            attention_weights.append(attn[0].cpu().numpy())
            input_token = tgt[0, t].unsqueeze(0)
    
    return np.array(attention_weights)

# Get attention for a sample
sample_text = "The quick brown fox"
noisy_text = noiser.add_noise(sample_text)

src = torch.tensor(tokenizer.encode(noisy_text)).unsqueeze(0).to(device)
tgt = torch.tensor(tokenizer.encode(sample_text)).unsqueeze(0).to(device)

attention = get_attention_weights(seq2seq_model, src, tgt)

# Plot attention heatmap
plt.figure(figsize=(12, 8))
plt.imshow(attention, aspect='auto', cmap='Blues')
plt.xlabel('Source Position (Noisy Text)')
plt.ylabel('Target Position (Clean Text)')
plt.title('Attention Weights Visualization')
plt.colorbar(label='Attention Weight')

# Add text labels
src_chars = list(noisy_text)
tgt_chars = list(sample_text)

plt.xticks(range(len(src_chars)), src_chars)
plt.yticks(range(len(tgt_chars)), tgt_chars)

plt.tight_layout()
plt.show()

print(f"Noisy:  {noisy_text}")
print(f"Clean:  {sample_text}")

## Part 26: Error Analysis

In [ ]:
def analyze_errors(model, test_texts: List[str], noiser: TextNoiser, is_transformer: bool = False, num_examples: int = 10):
    """Analyze common error patterns."""
    examples = []
    
    for clean_text in test_texts[:num_examples]:
        noisy_text = noiser.add_noise(clean_text)
        
        if is_transformer:
            denoised = denoise_text_transformer(model, noisy_text, tokenizer)
        else:
            denoised = denoise_text(model, noisy_text, tokenizer)
        
        cer = character_error_rate(denoised, clean_text)
        wer = word_error_rate(denoised, clean_text)
        
        examples.append({
            'clean': clean_text,
            'noisy': noisy_text,
            'denoised': denoised,
            'cer': cer,
            'wer': wer
        })
    
    # Sort by CER (worst first)
    examples.sort(key=lambda x: x['cer'], reverse=True)
    
    return examples

# Analyze errors
print("Error Analysis (Worst Cases):\n")
print("=" * 80)

errors = analyze_errors(transformer_model, val_texts, noiser, is_transformer=True, num_examples=20)

for i, ex in enumerate(errors[:5], 1):
    print(f"\nExample {i}:")
    print(f"Clean:    {ex['clean']}")
    print(f"Noisy:    {ex['noisy']}")
    print(f"Denoised: {ex['denoised']}")
    print(f"CER: {ex['cer']:.3f}, WER: {ex['wer']:.3f}")
    print("-" * 80)

## Part 27: Pre-trained Models - Theory

**Transfer Learning** with pre-trained models like **T5** (Text-to-Text Transfer Transformer) can significantly improve performance:

**Why Pre-trained Models?**
- Already learned language patterns from massive corpora
- Require less training data for fine-tuning
- Often achieve better performance with less compute

**T5 for Denoising:**
- Originally trained on denoising tasks (span corruption)
- Natural fit for text correction
- Frame as: "correct: [noisy text]" → "[clean text]"

**Fine-tuning Process:**
1. Load pre-trained T5 model (e.g., t5-small)
2. Prepare data with task prefix: "correct: "
3. Fine-tune on domain-specific noisy data
4. Evaluate on held-out test set

## Part 28: T5 Fine-tuning (Optional)

This section shows how to fine-tune T5 for denoising. **Note**: This requires the `transformers` library and may take longer to train.

In [ ]:
# Uncomment to install transformers if needed
# !pip install transformers

try:
    from transformers import T5Tokenizer, T5ForConditionalGeneration
    
    # Load pre-trained T5
    print("Loading T5 model...")
    t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')
    t5_model = T5ForConditionalGeneration.from_pretrained('t5-small').to(device)
    
    print(f"T5 model loaded with {count_parameters(t5_model):,} parameters")
    
    # Prepare data
    def prepare_t5_data(texts, noiser, tokenizer, max_length=128):
        """Prepare data for T5 training."""
        inputs = []
        targets = []
        
        for text in texts:
            noisy = noiser.add_noise(text)
            inputs.append(f"correct: {noisy}")
            targets.append(text)
        
        # Tokenize
        input_encodings = tokenizer(inputs, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        target_encodings = tokenizer(targets, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
        
        return input_encodings, target_encodings
    
    # Test T5 (before fine-tuning)
    test_text = "Machne lerning is fasinating"
    input_text = f"correct: {test_text}"
    input_ids = t5_tokenizer(input_text, return_tensors='pt').input_ids.to(device)
    
    outputs = t5_model.generate(input_ids, max_length=128)
    corrected = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\nT5 (pre-trained, no fine-tuning):")
    print(f"Input:  {test_text}")
    print(f"Output: {corrected}")
    
except ImportError:
    print("Transformers library not installed. Skipping T5 section.")
    print("Install with: pip install transformers")

## Part 29: Summary and Key Takeaways

**What We Learned:**

1. **Noise Types**: Character substitution, insertion, deletion, transposition
2. **Baseline Methods**: Edit distance with dictionary lookup
3. **Seq2Seq with Attention**: RNN encoder-decoder with attention mechanism
4. **Transformers**: Modern architecture with self-attention
5. **Evaluation**: CER and WER metrics for correction quality
6. **Pre-trained Models**: Transfer learning with T5

**Performance Comparison:**
- **Edit Distance**: Simple, interpretable, but limited to dictionary words
- **Seq2Seq**: Learns correction patterns, handles context
- **Transformer**: Better parallelization, often higher quality
- **Pre-trained (T5)**: Best performance, requires less training data

**Practical Considerations:**
- Data quality is crucial (garbage in, garbage out)
- Domain-specific noise patterns require domain-specific training
- Real-world noise is often more complex than synthetic
- Consider computational constraints for deployment

**Next Steps:**
- Try different noise levels and types
- Experiment with beam search decoding
- Fine-tune T5 on domain-specific data
- Explore other pre-trained models (BART, mT5)
- Add language model scoring for better corrections

## Part 30: Reflection Questions

Consider these questions to deepen your understanding:

1. **Architecture Choice**: When would you choose seq2seq over transformer, or vice versa?

2. **Noise Modeling**: How would you model real-world noise from OCR vs. keyboard typos?

3. **Evaluation**: Are CER/WER always the best metrics? What about semantic similarity?

4. **Training Data**: How much noisy data do you need? What's the right noise level?

5. **Production Deployment**: What challenges would you face deploying these models at scale?

6. **Attention Interpretation**: What do attention weights tell us about the model's correction strategy?

7. **Error Propagation**: How do errors compound when denoising multiple times?

8. **Domain Adaptation**: How would you adapt these models to medical/legal/technical text?

**Experiment Ideas:**
- Train with varying noise probabilities
- Try different tokenization (subword, word-level)
- Implement beam search decoding
- Add a language model scoring component
- Test on real OCR errors
- Explore semi-supervised approaches